### 전이학습(word2vec, Glove, FastText)
- word2vec
    - 주변단어를 통해 중심단어를 예측, 중심단어를 통해서 주변 단어를 예측 단어 벡터화
- Glove
    - 카운트 기반과 예측기반의 장점을 결합해서 단어의 동시등장 행렬을 분해
- FastText
    - 단어를 subword(n-gram) 단위로 쪼개서 oov(out-of-vocabulary)문제 해결

2. 정적 임베딩한계정 : 문맥 미반영(한 단어가 문맥에서 항상 동일한 벡터)
    - 동음이의어 처리 불가: 배를 먹가, 배를 타다, 배가 아프다에서 배는 모든 같은 벡터공간에 매핑

In [1]:
# 간단한 코퍼스를 활용 Word2Vec 학습, 단어간 유사도 확인
# !pip install gensim
from gensim.models import Word2Vec

In [2]:
# 토큰화된 문장
sentences = [
    ['I', 'love', 'natural', 'language', 'processing'],
    ['natural', 'language', 'processing', 'is', 'fun'],
    ['I', 'enjoy', 'learning', 'machine', 'learning']
]
# Word2Vec모델 학습(CBOW 기반)
w2v_model = Word2Vec(sentences,vector_size=10,window=2, min_count=1,epochs=10000)
print(f"language 벡터 : {w2v_model.wv['language']}")
print(f"natural, language의 유사도 : {w2v_model.wv.similarity('natural','language')}")
print(f"processing, language의 유사도 : {w2v_model.wv.similarity('processing','language')}")

language 벡터 : [ 0.46356866  0.7317976   0.73706156 -0.04372329  0.27976826 -0.24921551
  0.45000407  0.15153548 -1.0181676  -0.53059024]
natural, language의 유사도 : 0.919823408126831
processing, language의 유사도 : 0.9289067983627319


In [3]:
# 글로벌 동시 등장확률(Co occurence) 정보를 활용 : king - man + woman = queen 벡터연산이 작동하는지 확인
import gensim.downloader as api
print('Glove 모델 다운로드중.... 대략 66MB')
glove_model = api.load('glove-wiki-gigaword-50')
print('로딩완료\n')
print(f"king , queen 유사도 : {glove_model.similarity('king','queen')}")
print(f"man , woman 유사도 : {glove_model.similarity('man','woman')}")

Glove 모델 다운로드중.... 대략 66MB
로딩완료

king , queen 유사도 : 0.7839043140411377
man , woman 유사도 : 0.8860337734222412


In [4]:
# 벡터 연산  king-man+woman = queen
result = glove_model.most_similar(positive=['king','woman'], negative=['man'], topn=3)
print(f'king-man+woman 에 가장가까운단어 : {result}')

king-man+woman 에 가장가까운단어 : [('queen', 0.8523604869842529), ('throne', 0.7664334177970886), ('prince', 0.759214460849762)]


In [5]:
# FastText (OOV 문제 해결)
# 훈련데이터에 없거나 오타자에 대해 유연하게 동작하는지 확인
from gensim.models import FastText
# 실습데이터가 작아서  과적합을 피하기 위해 10000번학습
ft_model = FastText(sentences,vector_size=10,window=2,min_count=1,min_n=3, max_n=6, epochs=10000)
# 학습데이터에 존재하는 단어
print(f"language 벡터 : {ft_model.wv['languate']}")
print(f"학습되지 않은 단어 : languag : {ft_model.wv['languag']}")
print(f"langug, languge의 유사도 : {ft_model.wv.similarity('language', 'languag')}")

language 벡터 : [-0.26260483  0.35754687 -0.26880795  0.29179794  0.3724388   0.12110792
  0.19623452  0.30772692  0.13183883 -0.03056743]
학습되지 않은 단어 : languag : [-0.38566     0.5605096  -0.41307887  0.43644932  0.58965886  0.1577196
  0.2992876   0.47647923  0.20024978 -0.0366558 ]
langug, languge의 유사도 : 0.9998655319213867


In [6]:
# Static 임베딩의 한계점
# 동음의이어 처리 불가능  : 하나의 단어가 문맥에서 전혀 다른 의미로 사용될때 모델이 어떻게 반환하는지 실험
# bank(은행, 강둑)

vector_bank = glove_model['bank']
print(vector_bank.shape)
sim_money = glove_model.similarity('bank','money')
sim_river = glove_model.similarity('bank','river')
print(f'sim_money 유사도 : {sim_money}')
print(f'sim_river 유사도 : {sim_river}')
# 예상결과는 bank에 money하고 river의 의미가 짬봉된 vector 
# --> why 이미 수만은 문장을통해 bank money,  bank river의 관계를 학습

(50,)
sim_money 유사도 : 0.6360428929328918
sim_river 유사도 : 0.4142034947872162
